# Season-Aggregate Team Forecast -- CPU Smoke Test

**Track**: season-aggregate (deep history). Targets: `win_pct`,
`runs_scored_per_game`, `runs_allowed_per_game`, `team_ops`, `team_era` --
rate stats throughout, so strike seasons and 2020's 60-game season don't
need special-casing at this track (the Statcast-era track has its own
regime flags for 2020-and-later confounds; see `statcast_era_smoke.ipynb`).

**Purpose of this notebook**: prove the pipeline runs correctly on a tiny
slice (5 teams, a small `TeamPanelNet`, ~30 epochs) before scaling up to
all 30 teams / the full 1901-2025 history on Kaggle's T4x2 GPU. Model
*quality* is not the point here -- `notebooks/kaggle/season_aggregate_gpu/`
is where that's evaluated for real.

**Split** (mirrors the GPU notebook exactly, just on less data): train on
years strictly before 2015, hold out ALL of 2015-2025 as a genuine
backtest of whether pre-Statcast-era pattern-learning predicts the modern
game -- the model never sees the holdout years during training or
internal validation.


In [1]:

import json
import time
import numpy as np
import pandas as pd
import requests
import torch
from sklearn.metrics import mean_squared_error, r2_score

torch.manual_seed(42)
np.random.seed(42)

BASE_URL = "https://statsapi.mlb.com/api/v1"
HEADERS = {"User-Agent": "Mozilla/5.0 (MLB-Analytics-Dashboard-Telemetry/1.0; AustinKuo)"}


def safe_float(val, default=float("nan")):
    try:
        s = str(val).strip()
        return default if s in ("-.--", "---", "", "INF", "inf") else float(s)
    except (ValueError, TypeError):
        return default


def safe_int(val, default=0):
    try:
        return int(float(val))
    except (ValueError, TypeError):
        return default

print("Environment ready.")


Environment ready.


## Config

Smoke-scale: 5 teams (not all 30), a small hidden-layer net. The real
GPU notebook uses all 30 teams and a larger net -- see the plan's "Model
architecture" section for both scales' exact hyperparameters.


In [2]:

TRACK = "season_aggregate"
ENVIRONMENT = "local_smoke"

# 5 teams for the smoke test (real MLB ids) -- the GPU notebook uses all 30.
SMOKE_TEAMS = [
    {"team_id": 108, "abbreviation": "LAA"},
    {"team_id": 109, "abbreviation": "ARI"},
    {"team_id": 117, "abbreviation": "HOU"},
    {"team_id": 121, "abbreviation": "NYM"},
    {"team_id": 147, "abbreviation": "NYY"},
]
TEAM_FOUNDING_YEAR = {108: 1961, 109: 1998, 117: 1962, 121: 1962, 147: 1901}

TRAIN_START_YEAR = 1901
TRAIN_END_YEAR = 2014     # inclusive -- last pre-Statcast-era training year
HOLDOUT_START_YEAR = 2015
HOLDOUT_END_YEAR = 2025   # inclusive
FORECAST_END_YEAR = 2027  # one year past the plan's "past 2026" ask

TARGETS = ["win_pct", "runs_scored_per_game", "runs_allowed_per_game", "team_ops", "team_era"]

HIDDEN = (8, 8)
EMBED_DIM = 2
DROPOUT = 0.2
EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4

print(f"{len(SMOKE_TEAMS)} teams, train<= {TRAIN_END_YEAR}, holdout {HOLDOUT_START_YEAR}-{HOLDOUT_END_YEAR}")


5 teams, train<= 2014, holdout 2015-2025


## Data fetch

Reimplements `macroservice/teams.py`'s `get_team_season_stats` inline --
Kaggle kernels can't `pip install` this repo's local package, so every
notebook vendors its own minimal fetch logic (same reasoning as the
existing `mlb-aggregate-models-v2.ipynb`). Falls back to a synthetic
frame when the network is unavailable (offline validation, or a
rate-limited/empty response) so the rest of the pipeline always has
something to run against.


In [3]:

def fetch_team_season_stat_row(team_id, year):
    """One team-year's {wins, losses, gamesPlayed, runs_scored, runs_allowed,
    ops, era} or None if the API has nothing for that team/year (expansion
    teams before founding, work stoppage seasons the API omits entirely, etc).
    """
    try:
        hit = requests.get(f"{BASE_URL}/teams/{team_id}/stats",
                            params={"stats": "season", "group": "hitting", "season": year},
                            headers=HEADERS, timeout=15).json()
        pit = requests.get(f"{BASE_URL}/teams/{team_id}/stats",
                            params={"stats": "season", "group": "pitching", "season": year},
                            headers=HEADERS, timeout=15).json()
    except Exception:
        return None

    hit_splits = hit.get("stats", [{}])[0].get("splits", [])
    pit_splits = pit.get("stats", [{}])[0].get("splits", [])
    if not hit_splits or not pit_splits:
        return None
    h, p = hit_splits[0]["stat"], pit_splits[0]["stat"]
    games = safe_int(p.get("gamesPlayed"))
    if games == 0:
        return None
    return {
        "games": games,
        "wins": safe_int(p.get("wins")),
        "runs_scored": safe_int(h.get("runs")),
        "runs_allowed": safe_int(p.get("runs")),
        "team_ops": safe_float(h.get("ops")),
        "team_era": safe_float(p.get("era")),
    }


def synthetic_team_year(team_id, year, rng):
    """Demo row so the pipeline always runs offline/rate-limited -- a mild
    year-trend plus team-level noise, same spirit as the existing
    notebook's synthetic_frame().
    """
    trend = (year - 1901) / 125.0
    win_pct = float(np.clip(0.5 + 0.05 * rng.normal() + 0.02 * np.sin(team_id + trend * 10), 0.28, 0.72))
    return {
        "games": 162,
        "wins": round(win_pct * 162),
        "runs_scored": int(650 + 150 * trend + rng.normal(0, 40)),
        "runs_allowed": int(650 + 100 * trend + rng.normal(0, 40)),
        "team_ops": float(np.clip(0.68 + 0.05 * trend + rng.normal(0, 0.02), 0.55, 0.85)),
        "team_era": float(np.clip(4.2 - 0.3 * trend + rng.normal(0, 0.15), 2.8, 5.5)),
    }


def fetch_season_aggregate_frame(teams, start_year, end_year, use_synthetic_on_miss=True):
    rng = np.random.default_rng(42)
    rows = []
    for team in teams:
        for year in range(max(start_year, TEAM_FOUNDING_YEAR.get(team["team_id"], start_year)), end_year + 1):
            row = fetch_team_season_stat_row(team["team_id"], year)
            if row is None:
                if not use_synthetic_on_miss:
                    continue
                row = synthetic_team_year(team["team_id"], year, rng)
            row.update(team_id=team["team_id"], year=year)
            rows.append(row)
    df = pd.DataFrame(rows)
    df["win_pct"] = df["wins"] / df["games"]
    df["runs_scored_per_game"] = df["runs_scored"] / df["games"]
    df["runs_allowed_per_game"] = df["runs_allowed"] / df["games"]
    return df.sort_values(["team_id", "year"]).reset_index(drop=True)


raw = fetch_season_aggregate_frame(SMOKE_TEAMS, TRAIN_START_YEAR, HOLDOUT_END_YEAR)
print(f"Fetched {len(raw)} team-year rows across {raw['team_id'].nunique()} teams.")
raw.head()


Fetched 346 team-year rows across 5 teams.


## Feature engineering

Lag/rolling features per team, following `macroservice/features.py`'s
naming convention (`rolling_`) extended to annual cadence, plus a
`team_embedding_index` (0..N-1, a contiguous remap of the real MLB
`team_id`) for the model's embedding layer.


In [4]:

def add_lag_rolling_features(df, targets):
    df = df.sort_values(["team_id", "year"]).copy()
    for target in targets:
        grp = df.groupby("team_id")[target]
        df[f"lag_1_{target}"] = grp.shift(1)
        df[f"lag_3_avg_{target}"] = grp.shift(1).rolling(3, min_periods=1).mean().reset_index(level=0, drop=True)
        df[f"rolling_5yr_{target}"] = grp.shift(1).rolling(5, min_periods=1).mean().reset_index(level=0, drop=True)
    return df


team_ids_sorted = sorted(t["team_id"] for t in SMOKE_TEAMS)
TEAM_EMBED_INDEX = {tid: i for i, tid in enumerate(team_ids_sorted)}

featured = add_lag_rolling_features(raw, TARGETS)
featured["team_embedding_index"] = featured["team_id"].map(TEAM_EMBED_INDEX)
featured["year_norm"] = (featured["year"] - 1901) / 125.0
featured["team_age"] = featured["year"] - featured["team_id"].map(TEAM_FOUNDING_YEAR)

FEATURE_COLS = ["year_norm", "team_age"] + [
    f"{prefix}_{t}" for t in TARGETS for prefix in ("lag_1", "lag_3_avg", "rolling_5yr")
]
# First season per team has no lag history -- drop those rows rather than
# impute, since a team's true first-ever season isn't a meaningful lag target.
featured = featured.dropna(subset=FEATURE_COLS).reset_index(drop=True)
print(f"{len(featured)} feature-complete rows, {len(FEATURE_COLS)} numeric features.")


341 feature-complete rows, 17 numeric features.


## Split, model, training

Train on years < 2015 only; the internal validation split used for early
stopping is carved from *within* the pre-2015 training rows (last-2-years
per team), never from the 2015-2025 holdout -- the holdout must never
influence model selection or the backtest stops meaning anything.


In [5]:

import torch
import torch.nn as nn


class TeamPanelNet(nn.Module):
    """Feedforward net over engineered lag/rolling features, with team
    identity as a learned embedding and one shared trunk predicting all
    targets at once (a regularizer in itself at this row count, since the
    targets are correlated -- wins/runs-scored/runs-allowed/OPS/ERA all
    move together).
    """

    def __init__(self, n_teams, n_numeric_features, n_targets, embed_dim=8, hidden=(64, 32), dropout=0.2):
        super().__init__()
        self.team_embed = nn.Embedding(n_teams, embed_dim)
        layers, in_dim = [], embed_dim + n_numeric_features
        for h in hidden:
            layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(dropout)]
            in_dim = h
        self.trunk = nn.Sequential(*layers)
        self.head = nn.Linear(in_dim, n_targets)

    def forward(self, team_idx, x_numeric):
        z = torch.cat([self.team_embed(team_idx), x_numeric], dim=-1)
        return self.head(self.trunk(z))


def train_panel_net(model, X_team, X_num, y, epochs, lr=1e-3, weight_decay=1e-4,
                     val_team=None, val_num=None, val_y=None, patience=25):
    """Adam + early stopping on an internal validation set the caller
    carves out (never the real holdout -- see each notebook's split cell).
    Returns (train_loss_curve, val_loss_curve); leaves `model` trained
    in-place at its best-validation-loss state.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = torch.nn.MSELoss()
    train_curve, val_curve = [], []
    best_val, best_state, bad_epochs = float("inf"), None, 0
    has_val = val_team is not None and len(val_team) > 0

    for _ in range(epochs):
        model.train()
        optimizer.zero_grad()
        pred = model(X_team, X_num)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        train_curve.append(float(loss.item()))

        if has_val:
            model.eval()
            with torch.no_grad():
                val_loss = float(loss_fn(model(val_team, val_num), val_y).item())
            val_curve.append(val_loss)
            if val_loss < best_val:
                best_val, best_state, bad_epochs = val_loss, {k: v.clone() for k, v in model.state_dict().items()}, 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    break

    if has_val and best_state is not None:
        model.load_state_dict(best_state)
    return train_curve, val_curve


In [6]:

train_mask = featured["year"] <= TRAIN_END_YEAR
holdout_mask = (featured["year"] >= HOLDOUT_START_YEAR) & (featured["year"] <= HOLDOUT_END_YEAR)

train_df = featured[train_mask].copy()
holdout_df = featured[holdout_mask].copy()

# Internal validation: last 2 pre-2015 years per team (never touches holdout).
val_years_per_team = train_df.groupby("team_id")["year"].apply(lambda s: set(s.nlargest(2)))
is_val = train_df.apply(lambda r: r["year"] in val_years_per_team.get(r["team_id"], set()), axis=1)
fit_df, val_df = train_df[~is_val], train_df[is_val]

# Scale numeric features + targets on the *fit* split only.
feat_mean, feat_std = fit_df[FEATURE_COLS].mean(), fit_df[FEATURE_COLS].std().replace(0, 1.0)
y_mean, y_std = fit_df[TARGETS].mean(), fit_df[TARGETS].std().replace(0, 1.0)


def to_tensors(df):
    team_idx = torch.tensor(df["team_embedding_index"].to_numpy(), dtype=torch.long)
    x_num = torch.tensor(((df[FEATURE_COLS] - feat_mean) / feat_std).to_numpy(), dtype=torch.float32)
    y = torch.tensor(((df[TARGETS] - y_mean) / y_std).to_numpy(), dtype=torch.float32)
    return team_idx, x_num, y


fit_team, fit_num, fit_y = to_tensors(fit_df)
val_team, val_num, val_y = to_tensors(val_df) if len(val_df) else (None, None, None)

model = TeamPanelNet(len(team_ids_sorted), len(FEATURE_COLS), len(TARGETS), embed_dim=EMBED_DIM, hidden=HIDDEN, dropout=DROPOUT)
t0 = time.time()
train_curve, val_curve = train_panel_net(model, fit_team, fit_num, fit_y, epochs=EPOCHS, lr=LR,
                                          weight_decay=WEIGHT_DECAY, val_team=val_team, val_num=val_num,
                                          val_y=val_y, patience=10)
print(f"Trained {len(train_curve)} epochs in {time.time() - t0:.1f}s. Final train loss={train_curve[-1]:.4f}")


Trained 30 epochs in 10.4s. Final train loss=0.9939


## Holdout evaluation

Reports R²/RMSE in the same convention `chart.py` already uses for the
scikit-learn ensemble, so numbers are directly comparable across model
families -- plus a sanity linear-regression baseline (see the plan) fit
on the identical features, so a later read can tell whether the neural
net actually earned its complexity.


In [7]:

model.eval()
with torch.no_grad():
    holdout_team, holdout_num, _ = to_tensors(holdout_df.assign(**{t: 0.0 for t in TARGETS}))
    pred_scaled = model(holdout_team, holdout_num).numpy()
predictions = pred_scaled * y_std.to_numpy() + y_mean.to_numpy()

holdout_predictions, aggregate_holdout_metrics = [], {}
for i, target in enumerate(TARGETS):
    actual = holdout_df[target].to_numpy()
    pred = predictions[:, i]
    valid = ~np.isnan(actual)
    r2 = float(r2_score(actual[valid], pred[valid])) if valid.sum() >= 2 else None
    rmse = float(np.sqrt(mean_squared_error(actual[valid], pred[valid]))) if valid.sum() >= 2 else None
    aggregate_holdout_metrics[target] = {"r2": r2, "rmse": rmse, "n": int(valid.sum())}
    for (_, row), a, p in zip(holdout_df.iterrows(), actual, pred):
        holdout_predictions.append({"team_id": int(row["team_id"]), "year": int(row["year"]),
                                     "metric": target, "actual": None if np.isnan(a) else float(a),
                                     "predicted": float(p)})
    print(f"{target:>24s}  holdout R2={r2}  RMSE={rmse}  n={valid.sum()}")

from sklearn.linear_model import LinearRegression
baseline_comparison = {}
for target in TARGETS:
    lr_model = LinearRegression().fit(fit_df[FEATURE_COLS], fit_df[target])
    pred = lr_model.predict(holdout_df[FEATURE_COLS])
    actual = holdout_df[target].to_numpy()
    valid = ~np.isnan(actual)
    baseline_comparison[target] = {
        "r2": float(r2_score(actual[valid], pred[valid])) if valid.sum() >= 2 else None,
        "rmse": float(np.sqrt(mean_squared_error(actual[valid], pred[valid]))) if valid.sum() >= 2 else None,
    }
print("Baseline (plain linear regression) holdout R2:", {k: v["r2"] for k, v in baseline_comparison.items()})


                 win_pct  holdout R2=-0.12887043064214976  RMSE=0.07465480584828593  n=55
    runs_scored_per_game  holdout R2=-0.7084765081149869  RMSE=0.5935245981453215  n=55
   runs_allowed_per_game  holdout R2=-0.09227210978967926  RMSE=0.5404457824424411  n=55
                team_ops  holdout R2=-0.9279621345723919  RMSE=0.051200456654137486  n=55
                team_era  holdout R2=-5.50546659413625  RMSE=1.3070734124088697  n=55
Baseline (plain linear regression) holdout R2: {'win_pct': 0.2404525628775328, 'runs_scored_per_game': -0.17576248968933839, 'runs_allowed_per_game': 0.07283606804814935, 'team_ops': -0.24550949037384506, 'team_era': -0.29394126636299855}


## Forward forecast

Iterative/recursive rollout past the last known season: each future
year's lag features depend on the model's *own* prior-year prediction,
not ground truth (the same "extrapolate via `.predict()`" spirit as the
existing `regression.py::fit_and_forecast`, just iterative here since lag
features require it).


In [8]:

def forecast_forward(model, history_df, team_id, start_year, end_year):
    history = history_df[history_df["team_id"] == team_id].sort_values("year").copy()
    out = []
    for year in range(start_year, end_year + 1):
        row = {"team_id": team_id, "year": year, "year_norm": (year - 1901) / 125.0,
               "team_age": year - TEAM_FOUNDING_YEAR.get(team_id, year)}
        recent = history.tail(5)
        for target in TARGETS:
            row[f"lag_1_{target}"] = history[target].iloc[-1]
            row[f"lag_3_avg_{target}"] = history[target].tail(3).mean()
            row[f"rolling_5yr_{target}"] = recent[target].mean()
        x_num = torch.tensor([[(row[c] - feat_mean[c]) / feat_std[c] for c in FEATURE_COLS]], dtype=torch.float32)
        team_idx = torch.tensor([TEAM_EMBED_INDEX[team_id]], dtype=torch.long)
        with torch.no_grad():
            pred = (model(team_idx, x_num).numpy()[0] * y_std.to_numpy() + y_mean.to_numpy())
        for i, target in enumerate(TARGETS):
            row[target] = float(pred[i])
        out.append(row)
        history = pd.concat([history, pd.DataFrame([row])], ignore_index=True)
    return out


forward_forecasts = []
for team in SMOKE_TEAMS:
    for row in forecast_forward(model, featured, team["team_id"], HOLDOUT_END_YEAR + 1, FORECAST_END_YEAR):
        for target in TARGETS:
            forward_forecasts.append({"team_id": team["team_id"], "year": row["year"], "metric": target,
                                       "predicted": row[target], "ci_lower": None, "ci_upper": None})
print(f"{len(forward_forecasts)} forward forecast rows ({HOLDOUT_END_YEAR + 1}-{FORECAST_END_YEAR}).")


50 forward forecast rows (2026-2027).


## Results JSON export

Schema matches the plan's "Results JSON schema" section exactly -- this
is what a later Claude session reads to evaluate/refactor from, and what
`scripts/load_team_forecasts.py` reads to populate Postgres.


In [9]:

import datetime
import subprocess

try:
    git_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd="..").decode().strip()
except Exception:
    git_commit = None

run_stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y.%m.%d-%H%M")
results = {
    "schema_version": "1.0",
    "track": TRACK,
    "model_version": f"{TRACK}-{run_stamp}-smoke",
    "run_timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "environment": ENVIRONMENT,
    "git_commit": git_commit,
    "random_seed": 42,
    "hyperparameters": {"hidden_dims": list(HIDDEN), "embedding_dim": EMBED_DIM, "dropout": DROPOUT,
                         "lr": LR, "weight_decay": WEIGHT_DECAY, "epochs_trained": len(train_curve)},
    "targets": TARGETS,
    "feature_list": FEATURE_COLS,
    "teams": [{"team_id": t["team_id"], "abbreviation": t["abbreviation"],
               "embedding_index": TEAM_EMBED_INDEX[t["team_id"]]} for t in SMOKE_TEAMS],
    "training_window": {"start_year": TRAIN_START_YEAR, "end_year": TRAIN_END_YEAR},
    "holdout_window": {"start_year": HOLDOUT_START_YEAR, "end_year": HOLDOUT_END_YEAR},
    "regime_flags_used": [],
    "excluded_rows": [],
    "baseline_comparison": baseline_comparison,
    "aggregate_holdout_metrics": aggregate_holdout_metrics,
    "holdout_predictions": holdout_predictions,
    "forward_forecasts": forward_forecasts,
    "loss_curve": {"train": train_curve, "val": val_curve},
    "notes": "CPU smoke test on 5 teams -- not a quality signal, only a pipeline correctness check.",
}

import os as _os
_os.makedirs("results", exist_ok=True)
out_path = f"results/{results['model_version']}.json"
with open(out_path, "w") as fh:
    json.dump(results, fh, indent=2)
print("Wrote", out_path)


Wrote results/season_aggregate-2026.08.27-0045-smoke.json
